In [ ]:
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys; sys.path.append("../..")
from pipeline.utils.paper_style import RC

DATA_DIR = Path("../../datasets/findhr")
JUDGE = "ministral-3-14b"     # edit to switch judge
JUDGE_DIR = DATA_DIR / "judges" / JUDGE
print(f"judge: {JUDGE}")
PALETTE = ["#0072B2", "#009E73", "#E69F00"]  # Okabe-Ito: relevance grades 0/1/2
run_cols = [f"run_{i + 1}" for i in range(5)]
BAR_KW = dict(edgecolor="0.2", linewidth=0.6, zorder=3)   # fills go through solid()

sns.set_theme(style="whitegrid")   # before RC: set_theme overwrites rcParams
plt.rcParams.update(RC)

## 1. Load the ground truth

In [ ]:
gt = pd.read_csv(JUDGE_DIR / "llm_relevance.csv")
pairs = pd.read_csv(DATA_DIR / "applications/applications_short.csv")

print(f"scored: {len(gt):,} / {len(pairs):,} pairs ({len(gt) / len(pairs):.1%})")
print(f"jobs covered: {gt.id_j.nunique()} | cvs covered: {gt.id_c.nunique()}")
gt.head()

## 2. Join with the job / CV features

In [ ]:
jobs = pd.read_json(DATA_DIR / "source/filtered_jobs_dt.json")[
    ["id_j", "job_category_agg", "years_experience", "edu_eqf"]
].set_index("id_j")
cvs = pd.read_json(DATA_DIR / "source/filtered_cvs_dt.json")[
    ["id_c", "job_category_agg", "years_experience", "edu_eqf"]
].set_index("id_c")


def parse_years(value) -> float:
    """'1 - 2 years' -> 1.5, '+15 years' -> 15.0, 'Not specified' -> NaN."""
    nums = [int(x) for x in re.findall(r"\d+", str(value))]
    return float(np.mean(nums)) if nums else np.nan


def parse_edu(value) -> float:
    """'level 6' -> 6.0, 'Not specified' -> NaN."""
    match = re.search(r"\d+", str(value))
    return float(match.group()) if match else np.nan


job_feat = pd.DataFrame({
    "job_category": jobs.job_category_agg,
    "job_exp": jobs.years_experience.map(parse_years),
    "job_edu": jobs.edu_eqf.map(parse_edu),
})
cv_feat = pd.DataFrame({
    "cv_category": cvs.job_category_agg,
    "cv_exp": cvs.years_experience.map(parse_years),
    "cv_edu": cvs.edu_eqf.map(parse_edu),
})

analysis = (gt
    .join(job_feat, on="id_j")
    .join(cv_feat, on="id_c"))
analysis["category_match"] = analysis.job_category == analysis.cv_category
analysis["exp_gap"] = analysis.cv_exp - analysis.job_exp     # >0: candidate exceeds requirement
analysis["edu_gap"] = analysis.cv_edu - analysis.job_edu     # >0: candidate exceeds requirement
analysis.head()

## 3. Judge stability across the 5 runs

Each pair is judged 5 times at `temperature=0.6`; `median_relevance` is the median of the runs.

In [ ]:
run_std = gt[run_cols].std(axis=1)
agree_rate = (gt[run_cols].nunique(axis=1) == 1).mean()
print(f"all {5} runs agree: {agree_rate:.1%} of pairs")
print(f"mean within-pair std: {run_std.mean():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

nunique_counts = gt[run_cols].nunique(axis=1).value_counts().sort_index()
axes[0].bar(nunique_counts.index.astype(str), nunique_counts.values, color="#2a78d6", width=0.6)
axes[0].set_title("Distinct scores seen across the 5 runs", loc="left", fontsize=11, weight="bold")
axes[0].set_xlabel("# distinct values among run_1..run_5")
axes[0].set_ylabel("pair count")
axes[0].spines[["top", "right"]].set_visible(False)

sns.histplot(run_std, bins=20, color="#2a78d6", edgecolor="white", linewidth=0.3, ax=axes[1])
axes[1].set_title("Within-pair std across the 5 runs", loc="left", fontsize=11, weight="bold")
axes[1].set_xlabel("std(run_1..run_5)")
axes[1].set_ylabel("pair count")
axes[1].spines[["top", "right"]].set_visible(False)


## 4. Relevance by job category

In [ ]:
by_cat = analysis.groupby("job_category").median_relevance.mean().sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(by_cat.index, by_cat.values, color="#2a78d6")
ax.set_title("Mean relevance by job category", loc="left", fontsize=12, weight="bold")
ax.set_xlabel("mean median_relevance")
ax.spines[["top", "right"]].set_visible(False)


## 5. Judge confidence

- `agreement`: share of the 5 runs backing the final label (1.0 = unanimous, 0.8 = 4/5,
  0.6 = 3/5). Coarse: three values.
- `stability`: `1 - mean|run - label| / 2`. Same direction, but on the ordinal scale:
  runs `(2,2,2,2,1)` and `(2,2,2,2,0)` are both `agreement = 0.8`, and `stability` separates
  them (0.9 vs 0.8).

Both are recomputable from `run_1..run_5`, so they are never missing.

In [ ]:
assert {"agreement", "stability"} <= set(gt.columns), \
    "llm_relevance.csv predates the run-based confidence: re-score the pool with 2_1_llm_relevance.ipynb"
analysis["agreement"] = gt["agreement"]

non_unanimous = gt.agreement < 1.0     # the 5 runs did not all land on the same label

print(gt.groupby("agreement").median_relevance
        .value_counts(normalize=True).unstack().mul(100).round(1)
        .assign(pairs=gt.agreement.value_counts().sort_index(),
                share_of_pool=(gt.agreement.value_counts(normalize=True).sort_index() * 100).round(2)))
print(f"\nnon-unanimous: {non_unanimous.sum():,} pairs ({non_unanimous.mean():.1%} of the pool)")
print(f"stability: mean {gt.stability.mean():.3f} | "
      f"corr with agreement {gt[['agreement', 'stability']].corr().iloc[0, 1]:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

# (a) how the confidence mass is distributed
levels = gt.agreement.value_counts(normalize=True).sort_index() * 100
bars = axes[0].bar(levels.index.astype(str), levels.values, color="#2a78d6", width=0.6)
for bar, p in zip(bars, levels.values):
    axes[0].annotate(f"{p:.1f}%" if p >= 0.1 else f"{p:.2f}%",
                     (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                     textcoords="offset points", xytext=(0, 4), ha="center", fontsize=10, color="#0b0b0b")
axes[0].set_title("Judge confidence (run agreement)", loc="left", fontsize=11, weight="bold")
axes[0].set_xlabel("share of the 5 runs backing the label")
axes[0].set_ylabel("share of pairs (%)")
twin = axes[0].twinx()          # the finer-grained view of the same disagreement
sns.histplot(gt.stability, bins=20, color=PALETTE[2], alpha=0.35,
             edgecolor="white", linewidth=0.3, ax=twin)
twin.set_ylabel("pairs (stability)", color=PALETTE[2])
twin.grid(False)

# (b) what a non-unanimous pair looks like
mix = (analysis.assign(non_unanimous=analysis.agreement < 1.0)
       .groupby(["median_relevance", "category_match"]).non_unanimous.mean().unstack() * 100)
x = np.arange(len(mix))
axes[1].bar(x - 0.175, mix[False], 0.35, label="different category", color="#2a78d6")
axes[1].bar(x + 0.175, mix[True], 0.35, label="same category", color=PALETTE[1])
axes[1].set_xticks(x, labels=mix.index.astype(str))
axes[1].set_title("Where the uncertainty sits", loc="left", fontsize=11, weight="bold")
axes[1].set_xlabel("relevance grade")
axes[1].set_ylabel("non-unanimous pairs (%)")
axes[1].legend(frameon=False, fontsize=8)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)

In [ ]:
ground_truth = (gt[["id_j", "id_c", "median_relevance", "agreement", "stability"]]
    .rename(columns={"median_relevance": "relevance", "agreement": "confidence"}))
ground_truth.to_csv(JUDGE_DIR / "ground_truth.csv", index=False)
print(f"wrote ground_truth.csv | {ground_truth.shape} | grades {sorted(ground_truth.relevance.unique())}")
ground_truth.head()

In [ ]:
RESULTS_DIR = JUDGE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

gt_pairs = (gt[["id_j", "id_c", "median_relevance", "agreement", "stability"]]
    .rename(columns={"median_relevance": "grade", "agreement": "confidence"})
    .assign(category_match=analysis["category_match"].values))
assert len(gt_pairs) == len(gt) and not gt_pairs.isna().any().any()

gt_pairs.to_csv(RESULTS_DIR / "gt_pairs.csv", index=False)
print(f"wrote results/gt_pairs.csv | {gt_pairs.shape}")